In [9]:
!pip install matplotlib
!pip install seaborn 
!pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.2 MB 6.2 MB/s eta 0:00:02
   ----------------- ---------------------- 3.7/8.2 MB 9.8 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.2 MB 9.9 MB/s eta 0:00:01
   ------------------------------------- -- 7.6/8.2 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 8.7 MB/s  0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -- ------------------------------------- 2.1/36.6 MB 11.6 MB/s eta 0:00:03
   ---- ----------------------------------- 4.2/36.6 MB 9.9 MB/s eta 0:00:04
   ------ --------------------------------- 6.0/36.6 MB 9.5 MB/s eta 0:00:04
   -------- ------------------------------- 7.9/36.6 MB 9.4 MB/s eta 0:00:04
   ---------- ----------------------------- 9.7/36.6 MB 9.2 MB/s eta 0:00:03
   ------------- -------------------------- 12.1/36.6 MB 9.5 MB/s eta 0:00:03
   --------------- -

In [11]:
"""
Error analysis for NoAttn+Dup_1024 predictions CSV.
Expects columns: id, true_label, predicted_verdict, prob_False, prob_True, prob_Conflicting
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

CSV_PATH   = r"E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\my data\noattn_dup_1024_test_raw_confidence_scores.csv"      # <- change to your file
OUTPUT_DIR = "."
CLASSES    = ["False", "True", "Conflicting"]

df = pd.read_csv(CSV_PATH)

# ── 1. Basic split ────────────────────────────────────────────────────
df["correct"]      = df["true_label"] == df["predicted_verdict"]
df["confidence"]    = df[[f"prob_{c}" for c in CLASSES]].max(axis=1)
df["margin"]        = df.apply(
    lambda r: r[f"prob_{r['predicted_verdict']}"] - sorted(
        [r["prob_False"], r["prob_True"], r["prob_Conflicting"]])[-2],
    axis=1,
)  # gap between top-1 and top-2 prob — low margin = borderline call

errors  = df[~df["correct"]].copy()
correct = df[df["correct"]].copy()

print(f"Total samples : {len(df)}")
print(f"Correct       : {len(correct)}  ({len(correct)/len(df):.1%})")
print(f"Errors        : {len(errors)}  ({len(errors)/len(df):.1%})")

# ── 2. Error breakdown by true class ─────────────────────────────────
print("\nErrors by TRUE class:")
print(errors["true_label"].value_counts())

print("\nErrors by PREDICTED class:")
print(errors["predicted_verdict"].value_counts())

# ── 3. Confusion pairs (true -> predicted) ────────────────────────────
confusion_pairs = errors.groupby(["true_label", "predicted_verdict"]).size().sort_values(ascending=False)
print("\nTop confusion pairs (true -> predicted):")
print(confusion_pairs)

# ── 4. High-confidence wrong predictions — most interesting/worrying ──
high_conf_errors = errors.sort_values("confidence", ascending=False).head(20)
print("\nTop 20 high-confidence WRONG predictions (model was very sure and still wrong):")
print(high_conf_errors[["id", "true_label", "predicted_verdict", "confidence"]])
high_conf_errors.to_csv(f"{OUTPUT_DIR}/high_confidence_errors.csv", index=False)

# ── 5. Low-margin errors — borderline calls, near decision boundary ──
low_margin_errors = errors.sort_values("margin", ascending=True).head(20)
print("\nTop 20 LOW-MARGIN wrong predictions (close call between two classes):")
print(low_margin_errors[["id", "true_label", "predicted_verdict", "margin"]])

# ── 6. Save all errors sorted by confidence descending ────────────────
errors.sort_values("confidence", ascending=False).to_csv(
    f"{OUTPUT_DIR}/all_errors_sorted.csv", index=False)

# ── 7. Per-class precision/recall/F1 recomputed from CSV (sanity check) ─
from sklearn.metrics import classification_report, confusion_matrix
print("\n" + classification_report(df["true_label"], df["predicted_verdict"], labels=CLASSES))

# =======================================================================
# GRAPHS
# =======================================================================

# G1 — Confusion matrix (counts)
cm = confusion_matrix(df["true_label"], df["predicted_verdict"], labels=CLASSES)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/g1_confusion_matrix.png", dpi=120)
plt.close()

# G2 — Confusion matrix normalized by true-class row (shows per-class error rate)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix (row-normalized)")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/g2_confusion_matrix_normalized.png", dpi=120)
plt.close()

# G3 — Confidence distribution: correct vs incorrect (histogram/KDE)
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(correct["confidence"], color="green", label="Correct", kde=True, stat="density", alpha=0.4, ax=ax)
sns.histplot(errors["confidence"],  color="red",   label="Wrong",   kde=True, stat="density", alpha=0.4, ax=ax)
ax.set_xlabel("Predicted-class probability (confidence)"); ax.legend()
ax.set_title("Confidence Distribution: Correct vs Wrong")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/g3_confidence_dist.png", dpi=120)
plt.close()

# G4 — Per-class error rate bar chart
err_rate = (
    df.groupby("true_label")["predicted_verdict"]
      .apply(lambda s: (s != s.name).mean())
)
err_rate = err_rate.reindex(CLASSES)
fig, ax = plt.subplots(figsize=(6, 5))
err_rate.plot(kind="bar", color=["#d62728", "#2ca02c", "#1f77b4"], ax=ax)
ax.set_ylabel("Error rate"); ax.set_title("Error Rate by True Class")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/g4_error_rate_by_class.png", dpi=120)
plt.close()

# G5 — Margin (top1-top2 prob gap) distribution: correct vs wrong
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="correct", y="margin", ax=ax)


# ax.set_xticklabels(["Wrong", "Correct"])
ax.set_xticks([0, 1], labels=["Wrong", "Correct"])

ax.set_ylabel("Margin (top1 - top2 probability)")
ax.set_title("Decision Margin: Correct vs Wrong Predictions")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/g5_margin_boxplot.png", dpi=120)
plt.close()

# G6 — Per-sample stacked probability bars for worst errors (top 15 high-conf errors)
top_err = high_conf_errors.head(15).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(top_err))
colors = {"False": "#d62728", "True": "#2ca02c", "Conflicting": "#1f77b4"}
for c in CLASSES:
    ax.bar(top_err.index, top_err[f"prob_{c}"], bottom=bottom, label=c, color=colors[c])
    bottom += top_err[f"prob_{c}"].values
ax.set_xticks(top_err.index)
ax.set_xticklabels([f"{r.id}\n(true:{r.true_label})" for r in top_err.itertuples()], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Probability"); ax.legend(title="Class")
ax.set_title("Top 15 High-Confidence Errors — Probability Breakdown")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/g6_worst_errors_prob_breakdown.png", dpi=120)
plt.close()

# G7 — Scatter: confidence vs margin, colored by correctness (spot the ambiguous cluster)
fig, ax = plt.subplots(figsize=(7, 6))
for flag, color, label in [(True, "green", "Correct"), (False, "red", "Wrong")]:
    sub = df[df["correct"] == flag]
    ax.scatter(sub["confidence"], sub["margin"], alpha=0.4, s=15, color=color, label=label)
ax.set_xlabel("Confidence (top-1 prob)"); ax.set_ylabel("Margin (top1-top2)")
ax.set_title("Confidence vs Margin, by Correctness")
ax.legend()
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/g7_confidence_vs_margin_scatter.png", dpi=120)
plt.close()

print(f"\nAll graphs + CSVs saved to {OUTPUT_DIR}/")

Total samples : 2558
Correct       : 2024  (79.1%)
Errors        : 534  (20.9%)

Errors by TRUE class:
true_label
Conflicting    213
True           167
False          154
Name: count, dtype: int64

Errors by PREDICTED class:
predicted_verdict
False          257
Conflicting    178
True            99
Name: count, dtype: int64

Top confusion pairs (true -> predicted):
true_label   predicted_verdict
Conflicting  False                171
False        Conflicting           97
True         False                 86
             Conflicting           81
False        True                  57
Conflicting  True                  42
dtype: int64

Top 20 high-confidence WRONG predictions (model was very sure and still wrong):
      id   true_label predicted_verdict  confidence
632  649  Conflicting             False    0.946167
608  625  Conflicting             False    0.938629
657  674  Conflicting             False    0.934569
683  700  Conflicting             False    0.929553
721  738  Conflicti

In [ ]:
from scipy.stats import mannwhitneyu
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------
# Margin Analysis: Correct vs Wrong Predictions
# ---------------------------------------------------------------------

wrong_margin = errors["margin"]
correct_margin = correct["margin"]

print("=" * 70)
print("Decision Margin Statistics")
print("=" * 70)

stats = pd.DataFrame({
    "Wrong": wrong_margin.describe(),
    "Correct": correct_margin.describe()
})

display(stats)

# ---------------------------------------------------------------------
# Statistical significance
# ---------------------------------------------------------------------

u_stat, p_value = mannwhitneyu(
    wrong_margin,
    correct_margin,
    alternative="two-sided"
)

# Cohen's d
pooled_std = np.sqrt(
    (
        (len(wrong_margin)-1)*wrong_margin.var(ddof=1) +
        (len(correct_margin)-1)*correct_margin.var(ddof=1)
    ) /
    (len(wrong_margin)+len(correct_margin)-2)
)

cohens_d = (
    correct_margin.mean() - wrong_margin.mean()
) / pooled_std

print(f"Mann–Whitney U statistic : {u_stat:,.0f}")
print(f"P-value                  : {p_value:.3e}")
print(f"Cohen's d                : {cohens_d:.3f}")

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------

plot_df = pd.DataFrame({
    "Margin": df["margin"],
    "Prediction": np.where(df["correct"], "Correct", "Wrong")
})

plt.figure(figsize=(8,6))

sns.violinplot(
    data=plot_df,
    x="Prediction",
    y="Margin",
    inner=None,
    cut=0,
    alpha=0.4
)

sns.boxplot(
    data=plot_df,
    x="Prediction",
    y="Margin",
    width=0.25,
    showfliers=True
)

plt.ylabel("Decision Margin (Top-1 Probability − Top-2 Probability)", fontsize=12)
plt.xlabel("")
plt.title("Distribution of Decision Margins for Correct and Incorrect Predictions", fontsize=14)

plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("decision_margin_violin_boxplot.png", dpi=300)
plt.show()

Decision Margin Statistics


,Wrong,Correct
count,534.000000,2024.000000
mean,0.214660,0.413706
std,0.236194,0.233077
min,0.000212,0.000217
25%,0.036672,0.231110
50%,0.111873,0.390794
75%,0.316426,0.588616
max,0.895854,0.933645


Mann–Whitney U statistic : 268,209
P-value                  : 6.955e-72
Cohen's d                : 0.852


C:\Users\hp\AppData\Local\Temp\ipykernel_10876\11287258.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# New 

In [14]:
# Filter for high-confidence errors (above Q3 of confidence distribution)
q3_confidence = df["confidence"].quantile(0.75)

high_conf_wrong = errors[errors["confidence"] > q3_confidence].copy()
high_conf_wrong = high_conf_wrong.sort_values("confidence", ascending=False)

print("=" * 80)
print(f"HIGH-CONFIDENCE ERRORS (Confidence > Q3 = {q3_confidence:.4f})")
print("=" * 80)
print(f"\nTotal high-confidence wrong predictions: {len(high_conf_wrong)}")
print(f"\nDetails:\n")

# Display key columns
display_cols = ["id", "true_label", "predicted_verdict", "confidence", "prob_False", "prob_True", "prob_Conflicting"]
display(high_conf_wrong[display_cols])

# Save to CSV
high_conf_wrong[display_cols].to_csv(f"{OUTPUT_DIR}/high_confidence_errors_above_q3.csv", index=False)
print(f"\nSaved to: {OUTPUT_DIR}/high_confidence_errors_above_q3.csv")

HIGH-CONFIDENCE ERRORS (Confidence > Q3 = 0.7569)

Total high-confidence wrong predictions: 67

Details:



,id,true_label,predicted_verdict,confidence,prob_False,prob_True,prob_Conflicting
632,649,Conflicting,False,0.946167,0.946167,0.003521,0.050312
608,625,Conflicting,False,0.938629,0.938629,0.003485,0.057885
657,674,Conflicting,False,0.934569,0.934569,0.001838,0.063593
717,734,Conflicting,False,0.929553,0.929553,0.004126,0.066321
721,738,Conflicting,False,0.929553,0.929553,0.004126,0.066321
...,...,...,...,...,...,...,...
2273,2309,Conflicting,False,0.773898,0.773898,0.019924,0.206179
653,670,Conflicting,False,0.773640,0.773640,0.022787,0.203573
685,702,Conflicting,False,0.772785,0.772785,0.027156,0.200059
690,707,Conflicting,False,0.772785,0.772785,0.027156,0.200059



Saved to: ./high_confidence_errors_above_q3.csv
